In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.ensemble import IsolationForest

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [ ]:
# Load model outputs

trip_df = pd.read_csv("../reports/trip_risk_summary.csv")
driver_df = pd.read_csv("../reports/driver_risk_predictions.csv")

print("Trip summary shape:", trip_df.shape)
print("Driver prediction shape:", driver_df.shape)

trip_df.head()

In [ ]:
print("Trip Dataset Info")
display(trip_df.info())

print("\nDriver Dataset Info")
display(driver_df.info())

In [ ]:
data = trip_df.merge(driver_df, on="trip_id", how="left")

print("Merged dataset shape:", data.shape)

data.head()

In [ ]:
data["risk_score"] = data["risk_score_y"]
data["aggressive_probability"] = data["aggressive_probability_y"]

data.drop(
    ["risk_score_x","risk_score_y",
     "aggressive_probability_x","aggressive_probability_y"],
    axis=1,
    inplace=True
)

data.columns

In [ ]:
print("Dataset info")
display(data.info())

data.describe()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(data["risk_score"], bins=40, kde=True)

plt.title("Distribution of Trip Risk Scores")
plt.xlabel("Risk Score")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

ax = sns.countplot(
    data=data,
    x="risk_category",
    palette="viridis"
)

plt.title("Distribution of Risk Categories")
plt.xlabel("Risk Category")
plt.ylabel("Number of Samples")

# show numbers on bars
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [ ]:
print("Class Distribution:")
print(data["risk_category"].value_counts())

print("\nPercentage:")
print(data["risk_category"].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=data,
    x="context_cluster",
    y="risk_score"
)

plt.title("Risk Score across Driving Contexts")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(
    data=data,
    x="context_label"
)

plt.title("Trips per Driving Context")
plt.xticks(rotation=30)

plt.show()


# Feature correlation analysis
features = [

"aggression_score",
"control_instability",
"engine_stress_events",
"harsh_acc_events",
"harsh_brake_events",
"speed_stability",
"throttle_smoothness",
"acc_variability",
"jerk_intensity",
"risk_score"

]

plt.figure(figsize=(10,8))

sns.heatmap(
    data[features].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Feature Correlation Matrix")

plt.show()

In [ ]:
features = [

"aggression_score",
"control_instability",
"engine_stress_events",
"harsh_acc_events",
"harsh_brake_events",
"speed_stability",
"throttle_smoothness",
"acc_variability",
"jerk_intensity",
"risk_score"

]

plt.figure(figsize=(10,8))

sns.heatmap(
    data[features].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Feature Correlation Matrix")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=data,
    x="aggression_score",
    y="risk_score",
    hue="context_cluster"
)

plt.title("Aggression Score vs Risk")

plt.show()

In [ ]:
driver_features = data.groupby("trip_id").agg({

"risk_score":"mean",
"aggression_score":"mean",
"control_instability":"mean",
"engine_stress_events":"sum"

}).reset_index()

In [ ]:
scaler = StandardScaler()

X = scaler.fit_transform(
    driver_features.drop("trip_id", axis=1)
)

In [ ]:
pca = PCA(n_components=2)

components = pca.fit_transform(X)

driver_features["PC1"] = components[:,0]
driver_features["PC2"] = components[:,1]

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=driver_features,
    x="PC1",
    y="PC2",
    hue="risk_score",
    palette="coolwarm"
)

plt.title("Driver Behaviour Clusters")

plt.show()

In [ ]:
anomaly_features = data[[
"speed_mean",
"rpm_mean",
"throttle_mean",
"acceleration_mean"
]]

In [ ]:
iso = IsolationForest(contamination=0.03)

data["anomaly"] = iso.fit_predict(anomaly_features)

data["anomaly"] = data["anomaly"].map({

1:"Normal",
-1:"Anomaly"

})

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=data,
    x="speed_mean",
    y="rpm_mean",
    hue="anomaly"
)

plt.title("Detected Driving Anomalies")

plt.show()

In [ ]:
# convert prediction labels to numeric

risk_map = {
    "Safe": 0,
    "Dangerous": 1
}

data["risk_category_encoded"] = data["risk_category"].map(risk_map)

In [ ]:
print(data.columns)

In [ ]:
leaderboard = data.groupby("trip_id").agg({

"risk_score":"mean"

}).sort_values("risk_score", ascending=False)

leaderboard.head(10)

In [ ]:
data.to_csv("../outputs/final_driver_analysis.csv", index=False)

leaderboard.to_csv("../outputs/driver_risk_leaderboard.csv")

print("Results exported successfully")